# Figure 5

This notebook builds Figure 5: SLIDE-informed directed-evolution strategy selection on NK and empirical landscapes. Raw simulation products are loaded from `raw_data/`, processed panel data are written to `processed_data/`, and final outputs are saved to `figures/{pdf,eps,png}/`.


## Setup

Load the shared SLIDE simulation, processing, and plotting tools. The notebook caches processed Figure 5 payloads so expensive directed-evolution trajectory simulations do not need to be repeated unless `OVERWRITE_PROCESSED_PKL` is set to `True`.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
from tqdm.auto import tqdm

from slide import selection_function_library as slct
from slide.data_generation import EMPIRICAL_NAMES, RAW_FILENAMES, load_empirical_landscape, strategy_grid, uniform_start_locs
from slide.direvo_functions import (
    build_NK_landscape_function,
    build_empirical_landscape_function,
    build_mutation_function,
    build_selection_function,
    get_single_decay_rate,
    model_function,
    run_directed_evolution,
)
from slide.utils import get_figures_dir, get_processed_data_dir, get_raw_data_dir, load_pickle, save_pickle

OVERWRITE_PROCESSED_PKL: bool = False
SAVE_FIGURES: bool = True
SAVE_TYPE_LIST = ("pdf", "png", "eps")
PANEL_DPI = 350

# Directed-evolution trace simulations are the slow part of this notebook.
NK_TRACE_REPS = 100
EMPIRICAL_TRACE_STARTS = 10
EMPIRICAL_TRACE_REPS = 100
EMPIRICAL_TRACE_STEPS = 150
TRACE_SEED = 42

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
for figure_type in SAVE_TYPE_LIST:
    (FIGURES_DIR / figure_type).mkdir(parents=True, exist_ok=True)

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")


def save_panel_figure(fig: plt.Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save one figure panel in every configured output format."""
    if not SAVE_FIGURES:
        return
    for figure_type in SAVE_TYPE_LIST:
        fig.savefig(FIGURES_DIR / figure_type / f"{stem}.{figure_type}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: plt.Axes, letter: str) -> None:
    """Add the bold panel letter used in the paper figures."""
    ax.text(-0.16, 1.08, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top", ha="left")


In [ ]:
# Fast vectorised trajectory + landscape helpers used by the Figure 5 trace cells.
from slide.data_generation import (
    nk_landscape_arrays,
    best_variant_traces_nk,
    best_variant_traces_empirical,
)

## Figure 5 Files And Parameters

Figure 5 uses the NK strategy lookup grid, the `N=4, A=20` lookup products for empirical strategy selection, and the empirical decay/strategy products for GB1, TrpB, TEV, and ParD3.


In [ ]:
RAW_KEYS = {
    "nk_lookup": "nk_strategy_grid",
    "n4_decay": "nk_decay_N4_A20",
    "n4_strategy": "nk_strategy_N4_A20",
    "n3_decay": "nk_decay_N3_A20",
    "n3_strategy": "nk_strategy_N3_A20",
    "empirical_decay": [f"empirical_decay_{name}_uniform" for name in EMPIRICAL_NAMES],
    # Unified best-variant trajectory sweep: heat map (final slice) AND DE lines come from this one product.
    "empirical_traj": [f"empirical_strategy_traj_{name}" for name in EMPIRICAL_NAMES],
}

PROCESSED_FILES = {
    "strategy_lookup": "figure5_strategy_lookup_processed.pkl",
    "nk_examples": "figure5_nk_examples_processed.pkl",
    "empirical": "figure5_empirical_processed.pkl",
}

NK_EXAMPLE_PAIRS = {
    "B": {"label": "Smooth NK", "N": 45, "K": 1},
    "C": {"label": "Rugged NK", "N": 45, "K": 25},
}

BASELINE_STRATEGY = {"name": "Baseline", "base_chance": 0.0, "split": 1, "color": "tab:orange"}
SLIDE_STRATEGY = {"name": "SLIDE", "color": "tab:blue"}
HIGH_EXPLORATION_STRATEGY = {"name": "HE", "base_chance": None, "split": None, "color": "#2ca02c"}
OPTIMUM_STRATEGY = {"name": "Optimum", "color": "red"}

required_raw_keys = [RAW_KEYS["nk_lookup"], RAW_KEYS["n4_decay"], RAW_KEYS["n4_strategy"], RAW_KEYS["n3_decay"], RAW_KEYS["n3_strategy"]]
required_raw_keys += RAW_KEYS["empirical_decay"] + RAW_KEYS["empirical_traj"]
required_raw_paths = {key: RAW_DATA_DIR / RAW_FILENAMES[key] for key in required_raw_keys}
missing_raw_paths = {key: path for key, path in required_raw_paths.items() if not path.exists()}

if missing_raw_paths:
    print("Missing raw products needed by Figure 5:")
    for key, path in missing_raw_paths.items():
        print(f"  {key}: {path}")
else:
    print("All Figure 5 raw products are present.")

## Processing Helpers

These helpers keep the processing cells focused on the figure logic: loading raw products, reducing strategy sweeps into strategy-space heat maps, mapping decay-rate estimates to lookup-table strategies, and running baseline versus SLIDE directed-evolution traces.


In [ ]:
def processed_path(key: str) -> Path:
    """Return the processed-data path for one Figure 5 product key."""
    return PROCESSED_DATA_DIR / PROCESSED_FILES[key]


def load_or_build_processed(key: str, builder):
    """Load a processed Figure 5 product, or build and save it when absent."""
    path = processed_path(key)
    if path.exists() and not OVERWRITE_PROCESSED_PKL:
        print(f"Loaded {path.name}")
        return load_pickle(path)
    payload = builder()
    save_pickle(payload, path)
    print(f"Saved {path.name}")
    return payload


def require_raw_products(keys: list[str]) -> None:
    """Raise a clear error if any required raw Figure 5 product is missing."""
    missing = [key for key in keys if not (RAW_DATA_DIR / RAW_FILENAMES[key]).exists()]
    if missing:
        message = "Missing raw products. Run data_generation.ipynb first for: " + ", ".join(missing)
        raise FileNotFoundError(message)


def load_raw_payload(key: str) -> dict:
    """Load a raw payload by the registry key used in slide.data_generation."""
    return load_pickle(RAW_DATA_DIR / RAW_FILENAMES[key])


def mean_strategy_space(scores: np.ndarray, grid_size: int, *, layout: str) -> np.ndarray:
    """Reduce a raw strategy sweep to a split-by-base-chance performance matrix.

    Parameters
    ----------
    scores:
        Raw strategy scores from a Figure 5 sweep.
    grid_size:
        Number of split and base-chance options along the strategy grid.
    layout:
        `"split_base_reps"` for saved NK-grid items with axes `(split, base, replicate)`,
        or `"last_base_split"` for raw generator outputs whose final axes are `(base, split)`.

    Returns
    -------
    np.ndarray
        Two-dimensional matrix with rows ordered by splitting option and columns ordered by
        base-chance option.
    """
    arr = np.asarray(scores, dtype=float)
    if layout == "split_base_reps":
        if arr.shape[0] != grid_size or arr.shape[1] != grid_size:
            arr = arr.reshape(grid_size, grid_size, -1)
        return arr.mean(axis=tuple(range(2, arr.ndim)))
    if layout == "last_base_split":
        if arr.shape[-2:] != (grid_size, grid_size):
            arr = arr.reshape(-1, grid_size, grid_size)
        leading_axes = tuple(range(arr.ndim - 2))
        return arr.mean(axis=leading_axes).T
    raise ValueError(f"Unknown strategy-space layout: {layout}")


def strategy_index(strategy_space: np.ndarray) -> tuple[int, int]:
    """Return the row and column index of the best-performing strategy."""
    return tuple(int(v) for v in np.unravel_index(np.nanargmax(strategy_space), strategy_space.shape))


def strategy_choice_from_index(row: int, column: int, base_chances: np.ndarray, splits: np.ndarray, *, name: str, color: str) -> dict:
    """Convert a strategy matrix index into a labelled strategy dictionary."""
    return {
        "name": name,
        "split_index": int(row),
        "base_index": int(column),
        "split": int(splits[row]),
        "base_chance": float(base_chances[column]),
        "color": color,
    }


def strategy_index_from_values(split: int, base_chance: float, base_chances: np.ndarray, splits: np.ndarray) -> tuple[int, int]:
    """Find the nearest strategy grid index for a split/base-chance pair."""
    row = int(np.argmin(np.abs(np.asarray(splits, dtype=float) - float(split))))
    column = int(np.argmin(np.abs(np.asarray(base_chances, dtype=float) - float(base_chance))))
    return row, column


def grouped_strategy_summary(rho_values: np.ndarray, values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Group strategy predictions by rounded ruggedness for Figure 5A error bars."""
    rounded = np.round(np.asarray(rho_values, dtype=float), 1)
    unique = np.unique(rounded)
    means = []
    stds = []
    for value in unique:
        grouped = np.asarray(values, dtype=float)[rounded == value]
        means.append(float(np.nanmean(grouped)))
        stds.append(float(np.nanstd(grouped)))
    return unique, np.asarray(means), np.asarray(stds)


def normalized_gmu(decay: np.ndarray) -> np.ndarray:
    """Compute normalized squared-fitness decay, G_mu, from raw diffusion curves."""
    arr = np.asarray(decay, dtype=float)
    g_mu = np.nanmean(arr**2, axis=tuple(range(arr.ndim - 1)))
    return g_mu / g_mu[0]


def fit_decay_for_strategy(g_mu: np.ndarray, *, mutation_rate: float, num_steps: int) -> tuple[float, np.ndarray]:
    """Fit a one-rate exponential model to G_mu and return rho_2 plus the fitted curve."""
    x_vals = np.linspace(0, num_steps - 1, num_steps)
    decay_rate = get_single_decay_rate(g_mu, mut=mutation_rate, num_steps=num_steps)
    fitted = model_function(x_vals, *decay_rate, mut=mutation_rate)
    return float(decay_rate[0] / 2), np.asarray(fitted)


def nearest_lookup_strategy(rho: float, lookup: dict) -> dict:
    """Select the lookup-table strategy whose fitted ruggedness is closest to `rho`."""
    index = int(np.argmin(np.abs(np.asarray(lookup["rho_values"], dtype=float) - float(rho))))
    return {
        "name": "SLIDE",
        "split": int(lookup["optimal_splits"][index]),
        "base_chance": float(lookup["optimal_base_chances"][index]),
        "lookup_index": index,
        "lookup_rho": float(lookup["rho_values"][index]),
        "color": SLIDE_STRATEGY["color"],
    }


def strategy_threshold(base_chance: float, thresholds: np.ndarray, base_chances: np.ndarray) -> float:
    """Return the threshold paired with the nearest base-chance value."""
    index = int(np.argmin(np.abs(np.asarray(base_chances, dtype=float) - float(base_chance))))
    return float(np.asarray(thresholds, dtype=float)[index])


def run_split_strategy_trace(
    rng: jax.Array,
    fitness_function,
    *,
    n_sites: int,
    num_alleles: int,
    start: np.ndarray,
    split: int,
    base_chance: float,
    threshold: float,
    popsize: int,
    mutation_rate: float,
    num_steps: int,
) -> np.ndarray:
    """Run one split directed-evolution replicate and keep the winning subpopulation trajectory."""
    subpopsize = max(1, int(popsize) // int(split))
    initial_population = jnp.tile(jnp.asarray(start, dtype=jnp.int32)[None, :], (subpopsize, 1))
    selection_function = build_selection_function(
        slct.base_chance_threshold_select,
        {"threshold": float(threshold), "base_chance": float(base_chance)},
    )
    mutation_function = build_mutation_function(float(mutation_rate), int(num_alleles))
    split_keys = jr.split(rng, int(split))

    def single_split(split_rng):
        history = run_directed_evolution(
            split_rng,
            initial_population,
            selection_function,
            mutation_function,
            fitness_function=fitness_function,
            num_steps=int(num_steps),
        )[1]
        return history["fitness"].mean(axis=-1)

    split_curves = jax.vmap(single_split)(split_keys)
    winner = int(np.asarray(split_curves[:, -1]).argmax())
    return np.asarray(split_curves[winner])


def summarize_strategy_traces(curves_by_strategy: dict[str, list[np.ndarray]]) -> dict[str, dict[str, np.ndarray]]:
    """Convert replicate trajectories into mean and standard-deviation summaries."""
    summary = {}
    for name, curves in curves_by_strategy.items():
        arr = np.asarray(curves, dtype=float)
        summary[name] = {
            "mean": np.nanmean(arr, axis=0),
            "std": np.nanstd(arr, axis=0),
            "all": arr,
        }
    return summary


def strategy_marker_matrix(choice: dict, base_chances: np.ndarray, splits: np.ndarray) -> np.ndarray:
    """Create a sparse matrix marking one selected strategy in a strategy space."""
    matrix = np.zeros((len(splits), len(base_chances)), dtype=float)
    row, column = strategy_index_from_values(choice["split"], choice["base_chance"], base_chances, splits)
    matrix[row, column] = 1.0
    return matrix


## Raw Product Loading

Load only the raw products needed for Figure 5. The first execution will stop here if the necessary raw data have not yet been generated.


In [ ]:
require_raw_products(required_raw_keys)

nk_lookup_payload = load_raw_payload(RAW_KEYS["nk_lookup"])
n4_decay_payload = load_raw_payload(RAW_KEYS["n4_decay"])
n4_strategy_payload = load_raw_payload(RAW_KEYS["n4_strategy"])
empirical_decay_payloads = {name: load_raw_payload(f"empirical_decay_{name}_uniform") for name in EMPIRICAL_NAMES}
# Unified trajectory sweep per landscape: the heat map and the DE lines are both derived from this.
empirical_traj_payloads = {name: load_raw_payload(f"empirical_strategy_traj_{name}") for name in EMPIRICAL_NAMES}
empirical_landscapes = {name: load_empirical_landscape(name) for name in EMPIRICAL_NAMES}

print("NK lookup:", np.asarray(nk_lookup_payload["data"]).shape)
print("N4A20 decay:", np.asarray(n4_decay_payload["data"]).shape)
print("N4A20 strategy:", np.asarray(n4_strategy_payload["data"]).shape)
for name in EMPIRICAL_NAMES:
    print(name, "decay", np.asarray(empirical_decay_payloads[name]["data"]).shape, "traj", np.asarray(empirical_traj_payloads[name]["data"]).shape)


## Panel A Processing

Reduce the NK strategy sweep into a lookup table: for each `(N, K)` point, identify the strategy-space optimum and summarize the selected base chance and splitting over ruggedness bins.


In [ ]:
def build_strategy_lookup_payload():
    nk_lookup_raw = np.asarray(nk_lookup_payload["data"], dtype=float)
    params = nk_lookup_payload["params"]
    grid_size = int(params["strategy_grid_size"])
    base_chances = np.asarray(params["base_chances"], dtype=float)
    splits = np.asarray(params["splits"], dtype=int)
    nk_pairs = np.asarray(params["nk_pairs"], dtype=int)

    strategy_spaces = []
    rho_values = []
    per_pair_split = []
    per_pair_base = []
    for pair, item in zip(nk_pairs, nk_lookup_raw):
        N, K = int(pair[0]), int(pair[1])
        space = mean_strategy_space(item, grid_size, layout="split_base_reps")
        strategy_spaces.append(space)
        rho_values.append(float((K + 1) / N))
        row, column = strategy_index(space)
        per_pair_split.append(int(splits[row]))
        per_pair_base.append(float(base_chances[column]))
    strategy_spaces = np.asarray(strategy_spaces)
    rho_values = np.asarray(rho_values)
    per_pair_split = np.asarray(per_pair_split)
    per_pair_base = np.asarray(per_pair_base)

    # Clean lookup: average the strategy SURFACES within each rho bin, then take the optimum once.
    # Averaging surfaces is far more stable than averaging per-(N,K) argmaxes (which is just noise).
    rounded = np.round(rho_values, 1)
    grouped_rho = np.unique(rounded)
    lookup_base, lookup_split = [], []
    bc_means, bc_stds, split_means, split_stds = [], [], [], []
    for value in grouped_rho:
        mask = rounded == value
        bin_space = strategy_spaces[mask].mean(axis=0)
        row, column = strategy_index(bin_space)
        lookup_split.append(int(splits[row]))
        lookup_base.append(float(base_chances[column]))
        bc_means.append(float(per_pair_base[mask].mean()))
        bc_stds.append(float(per_pair_base[mask].std()))
        split_means.append(float(per_pair_split[mask].mean()))
        split_stds.append(float(per_pair_split[mask].std()))

    return {
        "data": {
            "rho_values": rho_values,
            "nk_pairs": nk_pairs,
            "strategy_spaces": strategy_spaces,
            "grouped_rho": grouped_rho,
            "lookup_base_chance": np.asarray(lookup_base),
            "lookup_split": np.asarray(lookup_split),
            "base_chance_mean": np.asarray(bc_means),
            "base_chance_std": np.asarray(bc_stds),
            "split_mean": np.asarray(split_means),
            "split_std": np.asarray(split_stds),
            "base_chances": base_chances,
            "splits": splits,
        },
        "params": params,
        "metadata": {"paper_reference": "Figure 5A", "description": "NK strategy lookup: per-rho-bin averaged-surface optimum (clean line) + per-(N,K) argmax scatter."},
    }


strategy_lookup_payload = load_or_build_processed("strategy_lookup", build_strategy_lookup_payload)
strategy_lookup = strategy_lookup_payload["data"]
print("Figure 5A lookup bins:", len(strategy_lookup["grouped_rho"]))


### Panel A Individual Export

Save the annotated Figure 5A lookup-table subfigure.


In [ ]:
def plot_strategy_lookup(ax: plt.Axes, lookup: dict, *, title: str = "Graphical look-up table") -> None:
    """Plot the Figure 5A lookup: per-rho optimal base chance and splitting (mean +/- std)."""
    color_base = "tab:orange"
    color_split = "tab:blue"
    rho = lookup["grouped_rho"]
    ax2 = ax.twinx()
    ax.errorbar(rho, lookup["base_chance_mean"], yerr=lookup["base_chance_std"], fmt="o--", capsize=3, color=color_base, label="Optimal base chance")
    ax2.errorbar(rho, lookup["split_mean"], yerr=lookup["split_std"], fmt="o--", capsize=3, color=color_split, label="Optimal splitting")
    ax.set_xlabel(r"$\rho_{NK}$")
    ax.set_ylabel(r"Predicted base chance $b$", color=color_base)
    ax2.set_ylabel("Predicted splitting", color=color_split)
    ax.tick_params(axis="y", labelcolor=color_base)
    ax2.tick_params(axis="y", labelcolor=color_split)
    ax.set_xlim(0.0, 1.05)
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=0.5)
    ax.set_title(title)
    handles1, labels1 = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(handles1 + handles2, labels1 + labels2, fontsize=7, frameon=True, loc="upper left")


fig, ax = plt.subplots(figsize=(3.5, 3.0), dpi=PANEL_DPI)
plot_strategy_lookup(ax, strategy_lookup, title="SLIDE NK strategy lookup")
add_panel_letter(ax, "A")
save_panel_figure(fig, "figure_5A")
plt.show()


## Panels B And C Processing

Use the NK lookup grid to choose smooth and rugged example strategy spaces, then run baseline and SLIDE-selected directed-evolution trajectories on matching NK landscapes. The strategy-space inserts are saved separately from the combined Figure 5 panel.


In [ ]:
def build_nk_examples_payload():
    params = nk_lookup_payload["params"]
    grid_size = int(params["strategy_grid_size"])
    base_chances = np.asarray(params["base_chances"], dtype=float)
    thresholds = np.asarray(params["thresholds"], dtype=float)
    splits = np.asarray(params["splits"], dtype=int)
    nk_pairs = [tuple(map(int, pair)) for pair in params["nk_pairs"]]
    lookup_raw = np.asarray(nk_lookup_payload["data"], dtype=float)
    pair_to_index = {pair: index for index, pair in enumerate(nk_pairs)}

    examples = {}
    for panel, config in NK_EXAMPLE_PAIRS.items():
        pair = (int(config["N"]), int(config["K"]))
        if pair not in pair_to_index:
            raise ValueError(f"NK example pair {pair} is not present in the lookup grid.")

        raw_index = pair_to_index[pair]
        strategy_space = mean_strategy_space(lookup_raw[raw_index], grid_size, layout="split_base_reps")
        optimum_row, optimum_column = strategy_index(strategy_space)
        optimum = strategy_choice_from_index(optimum_row, optimum_column, base_chances, splits, name="Optimum", color=OPTIMUM_STRATEGY["color"])
        slide = {**optimum, "name": "SLIDE", "color": SLIDE_STRATEGY["color"]}
        baseline_row, baseline_column = strategy_index_from_values(BASELINE_STRATEGY["split"], BASELINE_STRATEGY["base_chance"], base_chances, splits)
        baseline = strategy_choice_from_index(baseline_row, baseline_column, base_chances, splits, name="Baseline", color=BASELINE_STRATEGY["color"])
        he_row, he_column = strategy_index_from_values(int(splits[0]), float(base_chances[-1]), base_chances, splits)
        high_exploration = strategy_choice_from_index(he_row, he_column, base_chances, splits, name="HE", color=HIGH_EXPLORATION_STRATEGY["color"])

        N, K = pair
        # Pass the landscape to the fused kernel via traced arrays (one compile, reused across strategies).
        interaction_matrix, site_rng, offset_rng = nk_landscape_arrays(jr.PRNGKey(TRACE_SEED + raw_index), N, K)
        strategy_choices = {"Baseline": baseline, "SLIDE": slide, "Optimum": optimum, "HE": high_exploration}

        # Per-replicate random starts (shared across strategies) plus a distinct key per strategy.
        rep_keys = jr.split(jr.PRNGKey(TRACE_SEED + 1000 + raw_index), NK_TRACE_REPS)
        rep_splits = jax.vmap(lambda rep_key: jr.split(rep_key, 1 + len(strategy_choices)))(rep_keys)
        start_keys = rep_splits[:, 0]
        strategy_keys = rep_splits[:, 1:]
        start_coords = jax.vmap(lambda key: jr.randint(key, (N,), 0, int(params["A"])))(start_keys).astype(jnp.int32)

        curves_by_strategy = {}
        for position, (strategy_name, choice) in enumerate(strategy_choices.items()):
            threshold = strategy_threshold(choice["base_chance"], thresholds, base_chances)
            curves = best_variant_traces_nk(
                strategy_keys[:, position],
                start_coords,
                jnp.asarray(float(choice["base_chance"])),
                jnp.asarray(float(threshold)),
                interaction_matrix,
                site_rng,
                offset_rng,
                n_sites=N,
                num_alleles=int(params["A"]),
                split=int(choice["split"]),
                popsize=int(params["popsize"]),
                mutation_rate=float(params["mutation_rate"]) / N,
                num_steps=int(params["M"]),
            )
            curves_by_strategy[strategy_name] = np.asarray(curves)

        examples[panel] = {
            "label": config["label"],
            "N": N,
            "K": K,
            "rho_NK": float((K + 1) / N),
            "strategy_space": strategy_space,
            "base_chances": base_chances,
            "thresholds": thresholds,
            "splits": splits,
            "choices": strategy_choices,
            "traces": summarize_strategy_traces(curves_by_strategy),
            "generations": np.arange(int(params["M"])),
        }

    return {
        "data": examples,
        "params": {"trace_reps": NK_TRACE_REPS, "trace_seed": TRACE_SEED, **params},
        "metadata": {"paper_reference": "Figure 5B-C", "description": "NK directed-evolution trajectory examples (Baseline, SLIDE, Optimum, HE) with separate strategy-space inserts."},
    }


nk_examples_payload = load_or_build_processed("nk_examples", build_nk_examples_payload)
nk_examples = nk_examples_payload["data"]
for panel, payload in nk_examples.items():
    print(panel, payload["label"], "N", payload["N"], "K", payload["K"], "SLIDE", payload["choices"]["SLIDE"])

### Panels B And C Individual Exports

Save the NK fitness-trajectory panels and the separate strategy-space inserts requested for Figures 5B and 5C.


In [ ]:
TRACE_STRATEGY_ORDER = ["Baseline", "SLIDE", "Optimum", "HE"]


def plot_trace_summary(ax: plt.Axes, payload: dict, *, title: str, ylabel: str = "Fitness (a.u.)") -> None:
    """Plot Baseline, SLIDE, Optimum, and HE trajectory summaries with standard-deviation bands."""
    generations = payload["generations"]
    for name in TRACE_STRATEGY_ORDER:
        trace = payload["traces"][name]
        color = payload["choices"][name]["color"]
        ax.plot(generations, trace["mean"], color=color, linewidth=1.4, label=name)
        ax.fill_between(generations, trace["mean"] - trace["std"], trace["mean"] + trace["std"], color=color, alpha=0.18, linewidth=0)
    ax.set_title(title)
    ax.set_xlabel(r"Generations $M$")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, frameon=True)
    ax.grid(True, alpha=0.2)


def plot_strategy_space(ax: plt.Axes, payload: dict, *, title: str = "Strategy performance", vmin: float | None = None, vmax: float | None = None, value_scale: float = 1.0):
    """Plot a strategy-performance heat map with baseline, SLIDE, optimum, and HE markers.

    `value_scale` rescales the colour values (e.g. 100 to express max final fitness as a
    percentage of wild-type), keeping the heat-map colours identical and only changing the
    colourbar units so they match the directed-evolution trajectory axis.
    """
    space = np.asarray(payload["strategy_space"], dtype=float) * value_scale
    image = ax.imshow(space, aspect="auto", origin="upper", cmap="viridis", vmin=vmin, vmax=vmax)
    base_chances = np.asarray(payload["base_chances"], dtype=float)
    splits = np.asarray(payload["splits"], dtype=int)
    for name in ["Baseline", "SLIDE", "Optimum", "HE"]:
        choice = payload["choices"][name]
        row, column = strategy_index_from_values(choice["split"], choice["base_chance"], base_chances, splits)
        marker = "o" if name != "Optimum" else "*"
        size = 28 if name != "Optimum" else 55
        ax.scatter(column, row, s=size, color=choice["color"], marker=marker, edgecolors="black", linewidth=0.35, label=name)
    ax.set_xticks([0, len(base_chances) - 1])
    ax.set_xticklabels([f"{base_chances[0]:.2f}", f"{base_chances[-1]:.2f}"])
    ax.set_yticks([0, len(splits) - 1])
    ax.set_yticklabels([str(splits[0]), str(splits[-1])])
    ax.set_xlabel(r"Base chance $b$")
    ax.set_ylabel("No. subpopulations")
    ax.set_title(title)
    ax.legend(fontsize=6, frameon=True, loc="center left", bbox_to_anchor=(1.05, 0.5))
    return image


for panel in ["B", "C"]:
    payload = nk_examples[panel]
    fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
    plot_trace_summary(ax, payload, title=f"N = {payload['N']}, K = {payload['K']}")
    add_panel_letter(ax, panel)
    save_panel_figure(fig, f"figure_5{panel}")
    plt.show()

    fig, ax = plt.subplots(figsize=(2.0, 1.8), dpi=PANEL_DPI)
    image = plot_strategy_space(ax, payload, title=f"Figure 5{panel} insert")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Max final fitness (a.u.)")
    save_panel_figure(fig, f"figure_5{panel}_insert")
    plt.show()

## Panels D-G Processing

For each empirical landscape, fit the squared-fitness decay curve, select a SLIDE strategy from the `N=4, A=20` lookup, summarize the empirical strategy space, and run baseline versus SLIDE directed-evolution trajectories from 10 starting genotypes.


In [ ]:
def fit_decay_curve(raw_g_mu: np.ndarray, *, mutation_rate: float, num_steps: int) -> tuple[float, np.ndarray, np.ndarray]:
    """Fit G_mu and return rho_2 plus the asymptote-subtracted decay curve and fitted line.

    Matches the published figure: the fitted plateau (landscape-mean level) is subtracted and
    the result renormalised, so both the data and the fit decay from 1 down to 0.
    """
    decay_rate = get_single_decay_rate(raw_g_mu, mut=mutation_rate, num_steps=num_steps)
    asymptote = decay_rate[1]
    x_vals = np.linspace(0, num_steps - 1, num_steps)
    curve = np.asarray(raw_g_mu, dtype=float) - asymptote
    curve = curve / curve[0]
    fit = np.asarray(model_function(x_vals, *decay_rate, mut=mutation_rate), dtype=float) - asymptote
    fit = fit / fit[0]
    return float(decay_rate[0] / 2), curve, fit


def build_nk_lookup(decay_key: str, strategy_key: str) -> dict:
    """Build a decay-rate -> optimal-strategy lookup from one NK decay + strategy product pair.

    Used for both N=4 (GB1/TrpB/TEV) and N=3 (ParD3) so each empirical landscape is matched to a
    lookup of the correct dimensionality and grid size.
    """
    decay = np.asarray(load_raw_payload(decay_key)["data"], dtype=float)
    strategy_payload = load_raw_payload(strategy_key)
    strategy = np.asarray(strategy_payload["data"], dtype=float)
    decay_params = load_raw_payload(decay_key)["params"]
    strategy_params = strategy_payload["params"]
    grid_size = int(strategy_params["strategy_grid_size"])
    base_chances = np.asarray(strategy_params["base_chances"], dtype=float)
    splits = np.asarray(strategy_params["splits"], dtype=int)
    k_values = np.asarray(decay_params["K_values"], dtype=int)

    rho_values, optimal_splits, optimal_base_chances = [], [], []
    for k_index, _k in enumerate(k_values):
        g_mu = normalized_gmu(decay[k_index])
        rho, _curve, _fit = fit_decay_curve(g_mu, mutation_rate=float(decay_params["mutation_rate"]), num_steps=int(decay_params["M"]))
        space = mean_strategy_space(strategy[:, k_index], grid_size, layout="last_base_split")
        row, column = strategy_index(space)
        rho_values.append(rho)
        optimal_splits.append(int(splits[row]))
        optimal_base_chances.append(float(base_chances[column]))

    order = np.argsort(rho_values)
    return {
        "rho_values": np.asarray(rho_values)[order],
        "optimal_splits": np.asarray(optimal_splits)[order],
        "optimal_base_chances": np.asarray(optimal_base_chances)[order],
        "base_chances": base_chances,
        "splits": splits,
    }


def build_empirical_payload():
    # One lookup per dimensionality: N=4 (GB1/TrpB/TEV, 7x7) and N=3 (ParD3, 5x5).
    lookups = {
        4: build_nk_lookup("nk_decay_N4_A20", "nk_strategy_N4_A20"),
        3: build_nk_lookup("nk_decay_N3_A20", "nk_strategy_N3_A20"),
    }
    processed = {}
    for name in EMPIRICAL_NAMES:
        decay_payload = empirical_decay_payloads[name]
        traj_payload = empirical_traj_payloads[name]
        landscape = empirical_landscapes[name]
        params = traj_payload["params"]
        base_chances = np.asarray(params["base_chances"], dtype=float)
        thresholds = np.asarray(params["thresholds"], dtype=float)
        splits = np.asarray(params["splits"], dtype=int)

        # The ONE dataset: best-variant trajectory per (start, split, base) cell.
        traj = np.asarray(traj_payload["data"], dtype=float)        # (starts, splits, base, steps)
        heatmap_gen = int(params.get("heatmap_gen", traj.shape[-1]))
        heatmap_index = min(heatmap_gen, traj.shape[-1]) - 1        # heat map = calibration-point slice (e.g. gen 25)
        strategy_space = traj[..., heatmap_index].mean(axis=0)      # while the lines below use the full trajectory
        trace_mean = traj.mean(axis=0)                             # (splits, base, steps)
        trace_std = traj.std(axis=0)
        num_steps = traj.shape[-1]

        # Decay panel + rho_2 (independent decay product).
        raw_g_mu = normalized_gmu(decay_payload["data"])
        rho, g_mu, fit = fit_decay_curve(raw_g_mu, mutation_rate=float(decay_payload["params"]["mutation_rate"]), num_steps=int(decay_payload["params"]["M"]))

        slide_pick = nearest_lookup_strategy(rho, lookups[landscape.ndim])

        def make_choice(label: str, split: int, base_chance: float, color: str) -> dict:
            row, column = strategy_index_from_values(split, base_chance, base_chances, splits)
            choice = strategy_choice_from_index(row, column, base_chances, splits, name=label, color=color)
            choice["trace_mean"] = trace_mean[row, column]
            choice["trace_std"] = trace_std[row, column]
            return choice

        optimum_row, optimum_column = strategy_index(strategy_space)
        choices = {
            "Baseline": make_choice("Baseline", BASELINE_STRATEGY["split"], BASELINE_STRATEGY["base_chance"], BASELINE_STRATEGY["color"]),
            "SLIDE": make_choice("SLIDE", slide_pick["split"], slide_pick["base_chance"], SLIDE_STRATEGY["color"]),
            "Optimum": make_choice("Optimum", int(splits[optimum_row]), float(base_chances[optimum_column]), OPTIMUM_STRATEGY["color"]),
            "HE": make_choice("HE", int(splits[0]), float(base_chances[-1]), HIGH_EXPLORATION_STRATEGY["color"]),
        }
        traces = {name_: {"mean": choice["trace_mean"], "std": choice["trace_std"]} for name_, choice in choices.items()}

        processed[name] = {
            "g_mu": g_mu,
            "fit": fit,
            "decay_generations": np.arange(g_mu.shape[0]),
            "rho2_fit": rho,
            "strategy_space": strategy_space,
            "base_chances": base_chances,
            "thresholds": thresholds,
            "splits": splits,
            "choices": choices,
            "traces": traces,
            "trace_generations": np.arange(num_steps),
            "popsize": int(params["popsize"]),
            "decay_params": decay_payload["params"],
            "strategy_params": params,
        }

    return {
        "data": processed,
        "params": {"trace_steps": int(empirical_traj_payloads[EMPIRICAL_NAMES[0]]["params"]["M"]), "trace_seed": TRACE_SEED},
        "metadata": {"paper_reference": "Figure 5D-G", "description": "Empirical panels: heat map (final slice) and DE lines derived from one best-variant trajectory sweep; ParD3 handled as N=3."},
    }


empirical_payload = load_or_build_processed("empirical", build_empirical_payload)
empirical_data = empirical_payload["data"]
for name, payload in empirical_data.items():
    print(name, "rho2", f"{payload['rho2_fit']:.3f}", "SLIDE", payload["choices"]["SLIDE"]["split"], payload["choices"]["SLIDE"]["base_chance"], "popsize", payload["popsize"])

### Panels D-G Individual Exports

Save each empirical row as its own annotated subfigure: decay curve, strategy space, and directed-evolution trajectories.


In [ ]:
def plot_decay_panel(ax: plt.Axes, payload: dict, *, name: str) -> None:
    """Plot the empirical G_mu decay curve (asymptote-subtracted) and fitted line."""
    ax.scatter(payload["decay_generations"], payload["g_mu"], s=12, color="tab:blue", label=r"Mean $G_\mu$")
    ax.plot(payload["decay_generations"], payload["fit"], color="tab:blue", linewidth=1.2, label="Fit")
    ax.set_ylim(0, 1.08)
    ax.set_title(rf"{name} fitness decay, $\rho_2={payload['rho2_fit']:.2f}$")
    ax.set_xlabel(r"Generations $M$")
    ax.set_ylabel(r"Normalised $G_\mu$ decay")
    ax.legend(fontsize=7, frameon=True)
    ax.grid(True, alpha=0.2)


def plot_empirical_de_panel(ax: plt.Axes, payload: dict, *, name: str) -> None:
    """Plot Baseline, SLIDE, Optimum, and HE empirical directed-evolution trajectories as % of WT fitness."""
    generations = payload["trace_generations"]
    for strategy_name in TRACE_STRATEGY_ORDER:
        trace = payload["traces"][strategy_name]
        color = payload["choices"][strategy_name]["color"]
        mean = trace["mean"] * 100.0
        std = trace["std"] * 100.0
        ax.plot(generations, mean, color=color, linewidth=1.2, label=strategy_name)
        ax.fill_between(generations, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)
    ax.set_title(f"{name} directed evolution")
    ax.set_xlabel(r"Generations $M$")
    ax.set_ylabel("Fitness relative to WT (%)")
    ax.legend(fontsize=7, frameon=True)
    ax.grid(True, alpha=0.2)


# Empirical max-final-fitness is expressed as % of wild-type so the colourbar matches the trajectory axis.
EMPIRICAL_PERF_LABEL = "Max fitness rel. to WT (%)"

for letter, name in zip(["D", "E", "F", "G"], EMPIRICAL_NAMES):
    payload = empirical_data[name]
    fig, axes = plt.subplots(1, 3, figsize=(8.6, 2.1), dpi=PANEL_DPI, constrained_layout=True)
    plot_decay_panel(axes[0], payload, name=name)
    image = plot_strategy_space(axes[1], payload, title="Strategy performance", value_scale=100.0)
    plot_empirical_de_panel(axes[2], payload, name=name)
    add_panel_letter(axes[0], letter)
    fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04, label=EMPIRICAL_PERF_LABEL)
    save_panel_figure(fig, f"figure_5{letter}")
    plt.show()

## Visualisation

Assemble the full Figure 5. Panels B and C show only the NK directed-evolution trajectories here; their strategy-performance inserts are saved separately as `figure_5B_insert` and `figure_5C_insert`. The arrow annotations between empirical decay, strategy-space, and trajectory panels are intentionally omitted for now.


In [ ]:
strategy_lookup_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["strategy_lookup"])
nk_examples_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["nk_examples"])
empirical_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["empirical"])

strategy_lookup = strategy_lookup_payload["data"]
nk_examples = nk_examples_payload["data"]
empirical_data = empirical_payload["data"]


In [ ]:
labelsize = 8
ticksize = 6
titlesize = 9
legendsize = 7
dpi = 350
plt.rcParams["font.family"] = "DejaVu Sans"

fig = plt.figure(figsize=(9.3, 9.4), dpi=dpi, constrained_layout=True)
gs = GridSpec(5, 3, figure=fig, height_ratios=[1.15, 1.0, 1.0, 1.0, 1.0], width_ratios=[1.08, 0.72, 1.08])

axes = {
    "A": fig.add_subplot(gs[0, 0]),
    "B": fig.add_subplot(gs[0, 1]),
    "C": fig.add_subplot(gs[0, 2]),
}
empirical_axes = {}
for row_index, letter in enumerate(["D", "E", "F", "G"], start=1):
    empirical_axes[letter] = [fig.add_subplot(gs[row_index, column]) for column in range(3)]

plot_strategy_lookup(axes["A"], strategy_lookup, title="SLIDE NK strategy lookup")
add_panel_letter(axes["A"], "A")

for letter in ["B", "C"]:
    payload = nk_examples[letter]
    plot_trace_summary(axes[letter], payload, title=f"N = {payload['N']}, K = {payload['K']}")
    add_panel_letter(axes[letter], letter)

for letter, name in zip(["D", "E", "F", "G"], EMPIRICAL_NAMES):
    payload = empirical_data[name]
    row_axes = empirical_axes[letter]
    plot_decay_panel(row_axes[0], payload, name=name)
    image = plot_strategy_space(row_axes[1], payload, title="Strategy performance", value_scale=100.0)
    fig.colorbar(image, ax=row_axes[1], fraction=0.046, pad=0.04, label=EMPIRICAL_PERF_LABEL)
    plot_empirical_de_panel(row_axes[2], payload, name=name)
    add_panel_letter(row_axes[0], letter)

for ax in fig.axes:
    ax.tick_params(axis="both", which="major", labelsize=ticksize)
    ax.xaxis.label.set_size(labelsize)
    ax.yaxis.label.set_size(labelsize)
    ax.title.set_size(titlesize)

if SAVE_FIGURES:
    for figure_type in SAVE_TYPE_LIST:
        fig.savefig(FIGURES_DIR / figure_type / f"figure_5.{figure_type}", dpi=dpi, bbox_inches="tight")
plt.show()